# Energy Island System Optimization (MILP)

**Author:** Agus Samsudin  
**Domain:** Energy Systems Modelling &nbsp;·&nbsp; Optimization &nbsp;·&nbsp; Renewable Energy  
**Tools:** Python · Pyomo · GLPK · Pandas · NumPy · Matplotlib · Plotly

---

## Overview

This notebook develops a **Mixed Integer Linear Programming (MILP)** model
to optimize the generation mix and storage capacity of a renewable-based
**energy island** or **isolated microgrid** system.

The model simultaneously determines optimal **investment decisions** (installed
capacity of each technology) and **operational dispatch** (hourly generation
and storage scheduling) to meet system electricity demand at
**minimum total system cost** or **minimum CO₂ emissions**.

A **grid loss factor** is applied to scale the raw demand up to the gross
generation requirement, accounting for distribution and transmission losses
within the island network.

---

## System Technologies

### Variable Renewable Generation

| Technology | Description |
|---|---|
| **Wind** | Hourly generation profile scaled to 1 MW installed capacity |
| **Solar PV** | Hourly generation profile scaled to 1 MW installed capacity |

Variable renewable technologies provide low-carbon electricity but depend
on resource availability and cannot be dispatched on demand.

### Dispatchable Generation

| Technology | Dispatch mode | Description |
|---|---|---|
| **Biomass** | Non-flexible (must-run) | Combustion of solid biomass fuels; dispatch pinned to hourly profile × installed capacity |
| **Biogas** | Non-flexible (must-run) | Anaerobic digestion of organic waste; dispatch pinned to hourly profile × installed capacity |
| **Waste-to-Energy (WTE)** | Non-flexible (must-run) | Energy recovery from municipal solid waste; dispatch pinned to hourly profile × installed capacity |
| **Hydro** | Flexible dispatch | Run-of-river or reservoir hydro generation; free to dispatch 0 → profile × capacity |
| **Geothermal** | Non-flexible (must-run) | Baseload generation; dispatch pinned to hourly profile × installed capacity |
| **Gas turbine** | Flexible backup | Fast-response backup; highest cost and emissions |

> **Non-flexible dispatch:** Biomass, Biogas, Geothermal, and WTE are forced to
> dispatch exactly at `profile(t) × capacity` each hour via a zero-curtailment
> constraint. Unused capacity in flexible dispatchable or gas technologies is
> standby reserve — not curtailment.

### Energy Storage

| Technology | Typical Duration | Description |
|---|---|---|
| **BESS** | 2–6 hours | Battery Energy Storage System; fast response, flexible siting |
| **PHS** | 6–12 hours | Pumped Hydro Storage; large-scale, requires suitable topography |
| **Hydrogen** | 12–48+ hours | Long-duration storage via electrolysis and fuel cell/turbine |

Storage technologies improve system flexibility by balancing renewable
variability, shifting energy across time, and reducing curtailment.

---

## Objective

The model supports three optimization objectives:

| Objective | Description |
|---|---|
| **Lowest LCOE** | Minimises total annualised system cost (default) |
| **Lowest CO₂** | Minimises total annual CO₂ emissions via high carbon shadow price |
| **Most Diversified** | Forces all selected technologies to ≥ `DIVERSIFIED_MIN_MW`; objective remains lowest cost |

Total system cost includes:

- Capital investment cost (annualised via Capital Recovery Factor)
- Fixed operation and maintenance (O&M) cost
- Variable fuel cost (gas and dispatchable fuels)
- Soft penalty for renewable curtailment
- Soft penalty for unnecessary storage cycling

---

## Model Workflow

| Step | Description |
|:---:|---|
| **1** | Import required Python libraries |
| **2** | Load geographic project information |
| **3** | Configure system technologies and model parameters |
| **4** | Upload technology and resource parameter CSV |
| **5** | Upload hourly demand and generation time series |
| **6** | Build and solve the MILP optimization model |
| **7** | Export results to Excel and JSON |
| **8** | Analyse and visualize results |

---

## Installation

```bash
pip install pyomo pandas numpy matplotlib plotly openpyxl ipywidgets
```

**GLPK Solver:**
```bash
# Ubuntu / Debian
sudo apt install glpk-utils

# macOS
brew install glpk

# Windows — download from https://winglpk.sourceforge.net/
```

---

> **Disclaimer:** This model was developed for educational and research purposes.
> While every effort has been made to ensure accuracy, the author makes no guarantees
> regarding completeness or reliability of results. Use at your own risk.


---

## Step 1 — Required Libraries

The following Python packages are required to run this notebook:

| Library | Purpose |
|---|---|
| **Pandas** | Data loading, handling, and time-series processing |
| **NumPy** | Numerical operations and array handling |
| **Matplotlib** | Visualization of dispatch, capacity, LCOE, and residual load |
| **Pyomo** | Optimization modelling framework for the MILP formulation |
| **Plotly** | Interactive Sankey diagram and energy flow visualization |
| **ipywidgets** | Interactive file upload and configuration interface |
| **io** | Handling in-memory CSV data from widget file uploads |
| **openpyxl** | Excel export of hourly dispatch results |

The model uses the **GLPK (GNU Linear Programming Kit)** solver — an
open-source, freely available solver suitable for large linear and
mixed-integer programs.

A global matplotlib style is configured here so that all charts
share a consistent, professional appearance throughout the notebook.


In [ ]:
# ── Core libraries ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Optimization framework ─────────────────────────────────────────────────────
import pyomo.environ as pyo

# ── Interactive interface ──────────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go

# ── Global matplotlib style ────────────────────────────────────────────────────
plt.rcParams.update({
    # Typography
    'font.family':        'DejaVu Sans',
    'font.size':          10.5,
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
    'axes.titlepad':      12,
    'axes.labelsize':     10,
    'axes.labelpad':      6,
    'legend.fontsize':    9,
    'legend.title_fontsize': 9.5,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,

    # Spines
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.spines.left':   True,
    'axes.spines.bottom': True,
    'axes.linewidth':     0.6,

    # Grid
    'axes.grid':          True,
    'axes.grid.axis':     'y',
    'grid.color':         '#e5e5e5',
    'grid.linewidth':     0.7,
    'grid.linestyle':     '-',

    # Backgrounds
    'axes.facecolor':     '#fafafa',
    'figure.facecolor':   'white',

    # Lines
    'lines.linewidth':    1.8,
    'lines.markersize':   4,

    # Ticks
    'xtick.direction':    'out',
    'ytick.direction':    'out',
    'xtick.major.size':   3.5,
    'ytick.major.size':   0,
    'xtick.major.pad':    4,
    'ytick.major.pad':    4,
    'xtick.color':        '#555555',
    'ytick.color':        '#555555',

    # Misc
    'figure.dpi':         110,
    'savefig.dpi':        180,
    'savefig.bbox':       'tight',
    'axes.axisbelow':     True,
})


---

## Model Constants

All hard-coded assumptions are defined here as named constants.
This is the **only cell** that needs to be edited to change model assumptions —
no values are duplicated elsewhere in the notebook.

| Constant | Value | Source / Notes |
|---|---|---|
| `HOURS_PER_YEAR` | 8760 | Full-year hourly resolution |
| `GRID_LOSS_FACTOR` | 0.04 | Distribution loss factor — 4% of delivered energy lost in the island grid |
| `SOC_MIN_FRACTION` | 0.20 | Minimum state-of-charge — prevents deep discharge damage |
| `SOC_INIT_FRACTION` | 0.50 | Initial and final SoC for cyclic boundary condition |
| `CURTAILMENT_PENALTY` | 1 | €/MWh — small penalty to discourage oversized capacity with curtailment |
| `STORAGE_CHARGE_COST` | 5 | €/MWh — proxy cost to prevent unnecessary storage cycling |
| `CARBON_SHADOW_PRICE` | 10 000 | €/tCO₂ — shadow price applied in Lowest CO₂ objective mode |
| `DIVERSIFIED_MIN_MW`  | 1   | MW — minimum installed capacity per technology in Most Diversified mode |

### Notes on the Grid Loss Factor

The **grid loss factor** (`GRID_LOSS_FACTOR`) inflates the raw consumer demand
to the gross **generation requirement** that must be met at the point of injection:

$$P_{\text{gross}}(t) = P_{\text{demand}}(t) \times (1 + \text{GLF})$$

A 4% loss factor is a typical value for small isolated island networks.

### Notes on Soft Penalties

The **curtailment penalty** (`CURTAILMENT_PENALTY`) discourages the model
from building oversized capacity and simply curtailing the excess.
It is intentionally small (well below any technology LCOE) so it guides
dispatch without distorting investment decisions.

The **storage charge cost** (`STORAGE_CHARGE_COST`) prevents the solver
from cycling storage unnecessarily when there is no benefit.

The **carbon shadow price** (`CARBON_SHADOW_PRICE`) is applied in
"Lowest CO₂" objective mode by adding a very high cost weight to each
tonne of CO₂ emitted, effectively forcing the solver to minimise
emissions while still respecting investment feasibility constraints.

The **diversification minimum** (`DIVERSIFIED_MIN_MW`) sets the floor
capacity that each selected technology must install in "Most Diversified"
mode, ensuring no technology is excluded from the optimal portfolio.


In [ ]:
# ── Time resolution ───────────────────────────────────────────────────────────
HOURS_PER_YEAR       = 8760    # Full year hourly resolution

# ── Grid loss factor ──────────────────────────────────────────────────────────
GRID_LOSS_FACTOR     = 0.04    # Distribution loss fraction

# ── Storage operation limits ──────────────────────────────────────────────────
SOC_MIN_FRACTION     = 0.20    # Minimum state-of-charge (20 % DoD limit)
SOC_INIT_FRACTION    = 0.50    # Initial and final SoC (cyclic boundary condition)

# ── Objective soft penalties ──────────────────────────────────────────────────
CURTAILMENT_PENALTY  = 1       # €/MWh — penalty per MWh of curtailed renewable energy
STORAGE_CHARGE_COST  = 5       # €/MWh — proxy cost per MWh charged to storage
CARBON_SHADOW_PRICE  = 10_000  # €/tCO2 — penalises emissions in Lowest CO2 mode
DIVERSIFIED_MIN_MW   = 1       # MW — minimum installed capacity in Most Diversified mode


---

## Step 2 — Geographic Information

This section loads geographic information about the modeled energy system
location. The data provides useful **contextual documentation** for the
project and is displayed for reference.

Geographic coordinates are not directly used in the current optimization
model, but in future versions could be used to automatically retrieve
solar irradiance, wind resource, and hydrology data via APIs.

### PHS Feasibility Check

The `Max_Height_m` field is particularly important for **Pumped Hydro Storage
(PHS)** feasibility. PHS requires a minimum head difference of approximately
**300 m** between upper and lower reservoirs to be technically and economically
viable. The model will issue a warning if the uploaded location does not
meet this threshold.

**Expected file:** `geographic.csv` (upload via widget below)

**Required columns:**

| Column | Type | Description |
|---|---|---|
| `Name` | string | Location name (island, region, or project name) |
| `Latitude` | float | Decimal degrees north |
| `Longitude` | float | Decimal degrees east |
| `Max_Height_m` | float | Maximum terrain elevation difference (metres) — used for PHS feasibility |

**Example:**
```
Name,Latitude,Longitude,Max_Height_m
El Hierro,27.74,-18.03,1500
```


In [ ]:
class GeographicDescription:
    """Load and validate project geographic information."""

    REQUIRED_COLUMNS = {"Name", "Latitude", "Longitude", "Max_Height_m"}
    PHS_MIN_HEIGHT_M  = 300

    def __init__(self):
        self.data = None

    def _load(self, filepath):
        """Load CSV from filepath, validate, and store."""
        df = pd.read_csv(filepath)

        missing = self.REQUIRED_COLUMNS - set(df.columns)
        if missing:
            print(f"ERROR: Missing columns: {missing}")
            return

        for col in ["Latitude", "Longitude", "Max_Height_m"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        if df[["Latitude", "Longitude", "Max_Height_m"]].isnull().any().any():
            print("ERROR: Non-numeric values found in coordinate or height columns.")
            return

        self.data = df
        print("Geographic data loaded.")
        display(self.data)

        height = df["Max_Height_m"].iloc[0]
        if height < self.PHS_MIN_HEIGHT_M:
            print(f"  WARNING: Max_Height_m = {height} m is below the "
                  f"{self.PHS_MIN_HEIGHT_M} m threshold recommended for PHS feasibility.")
        else:
            print(f"  PHS feasibility: height = {height} m — threshold met.")

    def upload(self):
        style      = {"description_width": "140px"}
        path_input = widgets.Text(
            value="data/inputs/geographic_setup.csv",
            description="File path:",
            layout=widgets.Layout(width="500px"),
            style=style
        )
        btn    = widgets.Button(description="Load File", button_style="primary")
        output = widgets.Output()

        print("Geographic File Loader")
        print("Expected columns: Name, Latitude, Longitude, Max_Height_m")
        display(path_input, btn, output)

        def on_click(_):
            with output:
                output.clear_output()
                try:
                    self._load(path_input.value)
                except FileNotFoundError:
                    print(f"ERROR: File not found: {path_input.value}")
                except Exception as e:
                    print(f"ERROR: {e}")

        btn.on_click(on_click)


geo = GeographicDescription()
geo.upload()


---

## Step 3 — System Configuration

This section defines which technologies are included in the optimization
model and sets the key model parameters.

### Technology Categories

Technologies are grouped into three categories:

| Category | Technologies | Notes |
|---|---|---|
| **Generation Technologies** | Wind, Solar, Biomass, Biogas, Geothermal, Hydro, WTE | Each uses a normalised hourly profile; all require a matching time-series file |
| **Energy Storage** | BESS, PHS, Hydrogen | User sets maximum discharge duration (hours); power and energy capacity are co-optimised |
| **Balancing Technology** | Gas | Peaking unit providing firm capacity backup at high marginal cost |

### Storage Duration Parameter

The **max duration hours** parameter sets the upper bound on the
energy-to-power ratio (E/P ratio) for each storage technology.
The optimiser may select a shorter duration if it minimises cost.

### Demand Scaling

The **demand scale (%)** parameter multiplies every hourly value in the
uploaded demand CSV by a constant factor before it is passed to the model.
This allows scenario analysis without editing the underlying data file.

| Value | Effect |
|---|---|
| `100%` | Baseline demand — CSV loaded as-is (default) |
| `110%` | +10% demand growth scenario |
| `90%` | −10% demand reduction / efficiency scenario |
| `150%` | +50% electrification / EV uptake scenario |

$$P_{\text{scaled}}(t) = P_{\text{raw}}(t) \times \frac{\text{DemandScale}}{100}$$

### Discount Rate

The **discount rate** is used in the Capital Recovery Factor (CRF) to
annualise investment costs over each technology's economic lifetime.
A rate of 8% (0.08) is a commonly used default for energy planning
in developing regions and island systems.

Click **Confirm Setup** when all selections are complete.


In [ ]:
class SetupOptions:
    """Interactive system configuration widget."""

    STORAGE_DEFAULTS = {"BESS": 4, "PHS": 8, "Hydrogen": 24}

    def __init__(self):
        self.selected_gen       = []
        self.selected_storage   = []
        self.selected_balancing = []
        self.max_storage_hours  = {}
        self.objective          = "Lowest LCOE"
        self.discount_rate      = 0.08
        self.currency           = "€"
        self.demand_scale_pct   = 100.0

    def display(self):
        # ── Objective ─────────────────────────────────────────────────────
        self.objective_dd = widgets.Dropdown(
            options=["Lowest LCOE", "Lowest CO2", "Most Diversified"],
            description="Objective"
        )

        # ── Currency symbol text input ─────────────────────────────────────
        self.currency_input = widgets.Text(
            value="€",
            description="Currency:",
            placeholder="e.g. € $ £ RM Rp",
            layout=widgets.Layout(width="280px")
        )

        # ── Generation technologies ────────────────────────────────────────
        self.gen_boxes = [
            widgets.Checkbox(description=t)
            for t in ["Wind", "Solar", "Biomass", "Biogas", "Geothermal", "Hydro", "WTE"]
        ]

        # ── Storage technologies ───────────────────────────────────────────
        self.storage_items = []
        for name, default_hrs in self.STORAGE_DEFAULTS.items():
            cb  = widgets.Checkbox(description=name)
            hrs = widgets.FloatText(value=default_hrs, description="Max Hours",
                                    layout=widgets.Layout(width="200px"))
            self.storage_items.append({"name": name, "checkbox": cb,
                                        "hours_input": hrs, "row": widgets.HBox([cb, hrs])})

        # ── Balancing and financial parameters ────────────────────────────
        self.balancing_boxes    = [widgets.Checkbox(description="Gas")]
        self.discount_input     = widgets.FloatText(value=0.08, description="Discount Rate")
        self.demand_scale_input = widgets.FloatText(
            value=100.0,
            description="Demand Scale %:",
            style={"description_width": "140px"},
            layout=widgets.Layout(width="280px"),
            tooltip="100 = baseline, 110 = +10% growth, 90 = -10% reduction"
        )

        btn    = widgets.Button(description="Confirm Setup", button_style="success")
        output = widgets.Output()

        display(self.objective_dd)
        display(self.currency_input)
        print("\nVariable Renewable Generation:")
        for cb in self.gen_boxes:
            display(cb)
        print("\nEnergy Storage (set max duration hours):")
        for item in self.storage_items:
            display(item["row"])
        print("\nBalancing Technology:")
        for cb in self.balancing_boxes:
            display(cb)
        display(self.discount_input)
        display(self.demand_scale_input)
        display(btn)
        display(output)

        def confirm(_):
            self.selected_gen = [cb.description for cb in self.gen_boxes if cb.value]
            self.selected_storage  = []
            self.max_storage_hours = {}
            for item in self.storage_items:
                if item["checkbox"].value:
                    self.selected_storage.append(item["name"])
                    self.max_storage_hours[item["name"]] = item["hours_input"].value
            self.selected_balancing = [cb.description for cb in self.balancing_boxes if cb.value]
            self.objective          = self.objective_dd.value
            self.discount_rate      = self.discount_input.value
            self.currency           = self.currency_input.value.strip() or "€"
            self.demand_scale_pct   = self.demand_scale_input.value
            with output:
                output.clear_output()
                print("Setup confirmed.")
                print(f"  Objective      : {self.objective}")
                print(f"  Currency       : {self.currency}")
                print(f"  Generation     : {self.selected_gen}")
                print(f"  Storage        : {self.selected_storage}")
                print(f"  Max hours      : {self.max_storage_hours}")
                print(f"  Balancing      : {self.selected_balancing}")
                print(f"  Discount rate  : {self.discount_rate:.1%}")
                print(f"  Demand scale   : {self.demand_scale_pct:.1f}%")
                print(f"  Grid loss      : {GRID_LOSS_FACTOR:.1%}")

        btn.on_click(confirm)


setup = SetupOptions()
setup.display()


---

## Step 4 — Technology and Resource Parameters

This section loads the **techno-economic characteristics** of each technology
included in the optimization. These parameters define the cost structure,
physical limits, and environmental performance of each technology.

### Required File

Upload a CSV file with **one row per technology**. The `Sources` column must
exactly match the technology names selected in Step 3.

### Required Columns

| Column | Unit | Description |
|---|---|---|
| `Sources` | — | Technology name (must match Step 3 selections exactly) |
| `Max_Capacity_MW` | MW | Maximum installable capacity (resource or land constraint) |
| `Investment_per_MW` | €/MW | Capital cost per MW of installed power capacity |
| `O&M_per_MW_yr` | €/MW/yr | Annual fixed operation and maintenance cost |
| `Lifetime` | years | Economic asset lifetime (used in CRF calculation) |
| `CO2_per_MWh` | tCO₂/MWh | Direct CO₂ emissions per MWh of electricity generated |
| `Fuel_Cost` | €/MWhₜₕᵤₑₗ | Variable fuel cost per MWh of fuel consumed (0 for renewables) |
| `Efficiency` | 0–1 | Generator thermal efficiency or storage one-way efficiency |
| `Merit_Order` | integer | Dispatch priority (lower = dispatched first) |
| `Storage_MWh` | €/MWh | Capital cost per MWh of energy storage capacity (storage only; 0 for generators) |

### Notes

- For **renewable generators**, `Fuel_Cost` and `CO2_per_MWh` should be set to 0
- For **storage technologies**, both `Investment_per_MW` (power capacity cost)
  and `Storage_MWh` (energy capacity cost) must be provided
- `Efficiency` for storage represents the **one-way efficiency**;
  round-trip efficiency = efficiency²
- `Efficiency` for generators is used **only** in the economic calculation
  to convert fuel cost from €/MWh_fuel to €/MWh_electric — the generation
  profile files already represent electrical output and must not be scaled
  by efficiency again
- Technology parameter data can be sourced from
  [IRENA Cost Database](https://www.irena.org/costs),
  [NREL ATB](https://atb.nrel.gov/), or peer-reviewed literature


In [ ]:
class ResourceAssessment:
    """Load and validate technology and resource parameter CSV."""

    REQUIRED_COLUMNS = {
        "Sources", "Max_Capacity_MW", "Investment_per_MW",
        "O&M_per_MW_yr", "Lifetime", "CO2_per_MWh",
        "Fuel_Cost", "Efficiency", "Merit_Order", "Storage_MWh"
    }

    def __init__(self):
        self.data   = None
        self._cache = {}   # pre-built dict for O(1) parameter lookups

    def _load(self, filepath):
        """Load CSV from filepath, validate, and build lookup cache."""
        df = pd.read_csv(filepath)

        missing = self.REQUIRED_COLUMNS - set(df.columns)
        if missing:
            print(f"ERROR: Missing columns: {missing}")
            return

        if df["Sources"].isnull().any():
            print("ERROR: 'Sources' column contains blank values.")
            return

        numeric_cols = [
            "Max_Capacity_MW", "Investment_per_MW", "O&M_per_MW_yr",
            "Lifetime", "CO2_per_MWh", "Fuel_Cost", "Efficiency", "Storage_MWh"
        ]
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            if df[col].isnull().any():
                print(f"ERROR: Non-numeric values in column '{col}'.")
                return

        eff_bad = df[(df["Efficiency"] <= 0) | (df["Efficiency"] > 1)]
        if not eff_bad.empty:
            print(f"WARNING: Efficiency outside (0, 1]: {eff_bad['Sources'].tolist()}")

        self.data   = df
        self._cache = {row["Sources"]: row for _, row in df.iterrows()}
        print("Resource data loaded.")
        display(self.data)

    def upload(self):
        style      = {"description_width": "140px"}
        path_input = widgets.Text(
            value="data/inputs/resource_assessment.csv",
            description="File path:",
            layout=widgets.Layout(width="500px"),
            style=style
        )
        btn    = widgets.Button(description="Load File", button_style="primary")
        output = widgets.Output()

        print("Resource Assessment File Loader")
        print("Expected columns: Sources, Max_Capacity_MW, Investment_per_MW, ")
        print("  O&M_per_MW_yr, Lifetime, CO2_per_MWh, Fuel_Cost, Efficiency, Merit_Order, Storage_MWh")
        display(path_input, btn, output)

        def on_click(_):
            with output:
                output.clear_output()
                try:
                    self._load(path_input.value)
                except FileNotFoundError:
                    print(f"ERROR: File not found: {path_input.value}")
                except Exception as e:
                    print(f"ERROR: {e}")

        btn.on_click(on_click)

    def get(self, tech):
        """Return parameter row for a technology. Raises KeyError if not found."""
        if tech not in self._cache:
            raise KeyError(f"Technology '{tech}' not found in resource data.")
        return self._cache[tech]


resources = ResourceAssessment()
resources.upload()


---

## Step 5 — Time Series Data

The optimization model requires **hourly time series data** describing
electricity demand and the availability of each renewable resource.
All datasets must contain exactly **8760 hourly values** representing
one complete year.

### Required Datasets

| Dataset | Unit | Description |
|---|---|---|
| **Electricity demand** | MW | Hourly system demand (not normalised) — scaled by the Demand Scale % set in Step 3 |
| **Generation profile per technology** | MW / MWᴵⁿˢᵗˡˡᵉᵈ | Normalised output per 1 MW installed — one file per selected technology |

### Generation Profile Format

Generation profiles represent the **normalised hourly output** of each
technology per 1 MW of installed capacity (values between 0 and 1).
The profile already represents **electrical output** — generator efficiency
is embedded in the profile values and must **not** be applied again.

The MILP model scales these profiles by the optimised installed capacity
to compute available generation:

$$P_{\text{available}}(t) = \text{Cap} \times \text{profile}(t)$$

Generator efficiency ($\eta$) is used **only** in the economic calculation
to convert fuel cost from €/MWh$_{\text{fuel}}$ to €/MWh$_{\text{electric}}$:

$$c_{\text{mc}} = \frac{\text{Fuel\_Cost}}{\eta}$$

### Demand Scaling

The demand CSV is loaded at its original values and validated first.
The **Demand Scale %** configured in Step 3 is then applied to produce
the scaled demand array used by the model. The loader prints the raw
and scaled peak and annual totals for confirmation.

### Data Sources

| Source | Data Available |
|---|---|
| [Renewables.ninja](https://www.renewables.ninja) | Wind and solar PV generation profiles |
| [Global Solar Atlas](https://globalsolaratlas.info) | Solar irradiance and PV yield data |
| [PVGIS](https://re.jrc.ec.europa.eu/pvg_tools/) | European Commission PV estimation tool |
| [ERA5 / Copernicus](https://cds.climate.copernicus.eu) | Global climate reanalysis data |
| [IRENA Resource Map](https://resourceirena.irena.org) | Renewable resource atlases |

### Validation

Each uploaded file is automatically checked for:
- Correct row count (exactly 8760 rows)
- Absence of NaN or missing values
- Non-negative values throughout


In [ ]:
class TimeSeriesData:
    """Load and validate hourly time series for demand and generation."""

    def __init__(self, setup):
        self.setup      = setup
        self.generation = {}    # dict: tech_name -> np.array(8760,)
        self.demand     = None  # np.array(8760,)

    @staticmethod
    def _validate(arr, label):
        """Validate length, NaN, and sign. Raises ValueError on failure."""
        # Resolve expected length from global constant; fall back to 8760 if
        # the constants cell has not yet been executed in this kernel session.
        try:
            expected = HOURS_PER_YEAR
        except NameError:
            expected = 8760
        if len(arr) != expected:
            raise ValueError(
                f"{label}: expected {expected} rows, got {len(arr)}.")
        if np.isnan(arr).any():
            raise ValueError(f"{label}: contains NaN values.")
        if (arr < 0).any():
            raise ValueError(f"{label}: contains negative values.")

    @staticmethod
    def _load_csv(filepath, label):
        """Load a single-column CSV from filepath and return numpy array."""
        arr = pd.read_csv(filepath).iloc[:, 0].astype(float).values
        TimeSeriesData._validate(arr, label)
        print(f"  {label}: {len(arr)} values loaded.")
        return arr

    def upload_generation(self):
        """Display one file path input per selected generation technology."""
        style   = {"description_width": "160px"}
        output  = widgets.Output()
        inputs  = {}

        for tech in self.setup.selected_gen:
            default = f"data/time_series/{tech.lower()}_prod.csv"
            inp = widgets.Text(
                value=default,
                description=f"{tech} profile:",
                layout=widgets.Layout(width="500px"),
                style=style
            )
            inputs[tech] = inp
            display(inp)

        btn = widgets.Button(description="Load All Generation Profiles", button_style="primary")
        display(btn, output)

        def on_click(_):
            with output:
                output.clear_output()
                for tech_name, inp in inputs.items():
                    try:
                        self.generation[tech_name] = self._load_csv(inp.value, tech_name)
                    except FileNotFoundError:
                        print(f"ERROR: File not found: {inp.value}")
                    except ValueError as e:
                        print(f"ERROR: {e}")
                    except Exception as e:
                        print(f"ERROR ({tech_name}): {e}")
                loaded = list(self.generation.keys())
                if loaded:
                    print(f"\nLoaded: {loaded}")

        btn.on_click(on_click)

    def upload_demand(self):
        """Display a file path input for the demand profile."""
        style      = {"description_width": "140px"}
        path_input = widgets.Text(
            value="data/time_series/demand.csv",
            description="Demand profile:",
            layout=widgets.Layout(width="500px"),
            style=style
        )
        btn    = widgets.Button(description="Load Demand", button_style="primary")
        output = widgets.Output()

        display(path_input, btn, output)

        def on_click(_):
            with output:
                output.clear_output()
                try:
                    raw   = self._load_csv(path_input.value, "Demand")
                    scale = self.setup.demand_scale_pct / 100.0
                    self.demand = raw * scale
                    if scale == 1.0:
                        print(f"  Demand scaling  : 100.0% — no adjustment")
                    else:
                        print(f"  Demand scaling  : {self.setup.demand_scale_pct:.1f}% (×{scale:.4f})")
                        print(f"  Peak   (raw → scaled) : {raw.max():.1f} MW  →  {self.demand.max():.1f} MW")
                        print(f"  Annual (raw → scaled) : {raw.sum()/1e3:.1f} GWh  →  {self.demand.sum()/1e3:.1f} GWh")
                except FileNotFoundError:
                    print(f"ERROR: File not found: {path_input.value}")
                except ValueError as e:
                    print(f"ERROR: {e}")
                except Exception as e:
                    print(f"ERROR: {e}")

        btn.on_click(on_click)


ts = TimeSeriesData(setup)

print("Generation Profiles")
print("Edit paths below to match your data/time_series/ folder, then click Load.")
ts.upload_generation()

print("\nDemand Profile")
ts.upload_demand()


---

## Step 6 — Optimization Model (MILP)

The system is optimized using a **Mixed Integer Linear Programming (MILP)**
formulation implemented in [Pyomo](http://www.pyomo.org/) and solved
with the open-source **GLPK** solver.

### Decision Variables

**Investment variables** (capacity sizing):

| Variable | Unit | Description |
|---|---|---|
| `Cap[tech]` | MW | Installed capacity of each generation technology |
| `StoragePowerCap[s]` | MW | Installed power capacity of each storage technology |
| `StorageEnergyCap[s]` | MWh | Installed energy capacity of each storage technology |

**Operational variables** (dispatch scheduling):

| Variable | Unit | Description |
|---|---|---|
| `RenGen[tech, t]` | MW | Generation output of each non-gas technology at each hour |
| `Curtail[tech, t]` | MW | Curtailed generation at each hour (zero for non-flexible techs) |
| `GasGen[t]` | MW | Gas turbine output at each hour (if selected) |
| `Charge[s, t]` | MW | Storage charging power at each hour |
| `Discharge[s, t]` | MW | Storage discharging power at each hour |
| `SOC[s, t]` | MWh | Storage state of charge at each hour |

### Objective Function

**Lowest LCOE mode** — minimise total annualised system cost:

$$\min \sum_{g} \text{Cap}_g \cdot (K_g \cdot \text{CRF}_g + O\&M_g)
+ \sum_{s} \text{StorPow}_s \cdot (K_s^{\text{MW}} \cdot \text{CRF}_s + O\&M_s)
+ \sum_{s} \text{StorEne}_s \cdot K_s^{\text{MWh}} \cdot \text{CRF}_s
+ \text{fuel costs}
+ \text{curtailment penalty}
+ \text{storage cycling cost}$$

where the **Capital Recovery Factor** annualises investment over asset lifetime:

$$\text{CRF} = \frac{r(1+r)^n}{(1+r)^n - 1}$$

**Lowest CO₂ mode** — adds a large carbon shadow price to marginal costs,
heavily penalising CO₂-intensive technologies:

$$c_{\text{mc}}^{g} \leftarrow c_{\text{mc}}^{g} + \text{CO}_{2,g} \times \text{CSP}$$

where CSP = 10 000 €/tCO₂ is the carbon shadow price.

**Most Diversified mode** — keeps the standard lowest-cost objective but enforces
a mandatory minimum installed capacity of `DIVERSIFIED_MIN_MW` on every technology
selected in Step 3. The solver remains free to invest beyond this floor wherever
economically justified, but cannot exclude any checked technology.

### Key Constraints

| Constraint | Description |
|---|---|
| **Power balance** | Generation + discharge + gas = gross demand + charge (every hour) |
| **Capacity limits** | `RenGen ≤ Cap × profile` (availability upper bound) |
| **Non-flexible floor** | `Curtail[tech, t] == 0` for Biomass, Biogas, Geothermal, WTE — dispatch pinned to profile × capacity |
| **Max capacity bounds** | `Cap ≤ Max_Capacity_MW` (resource constraint) |
| **Storage power limits** | Charge and discharge ≤ power capacity |
| **SoC dynamics** | $\text{SoC}_{s,t} = \text{SoC}_{s,t-1} + \eta \cdot \text{Charge}_{s,t} - \text{Discharge}_{s,t} / \eta$ |
| **SoC limits** | $0.2 \cdot E_s \leq \text{SoC}_{s,t} \leq E_s$ |
| **Duration limits** | $E_s \leq P_s \cdot \text{MaxHours}_s$ |
| **Cyclic SoC** | End-of-year SoC = initial SoC (50% of energy capacity) |
| **Renewable charging** | Storage can only charge from renewable surplus (no gas → storage) |
| **Grid loss factor** | $P_{\text{gross}}(t) = P_{\text{demand}}(t) \times (1 + \text{GLF})$ |


In [ ]:
class IslandEnergyMILP:
    """
    MILP investment + dispatch model for an island / microgrid energy system.

    Technology parameters are retrieved via resources.get(tech) — a pre-built
    dict lookup — rather than re-filtering the DataFrame on every constraint call.
    """

    # ── Non-flexible dispatch technologies — must follow profile exactly ────
    NON_FLEXIBLE = {"Biomass", "Biogas", "Geothermal", "WTE"}

    def __init__(self, setup, resources, ts):
        self.setup         = setup
        self.resources     = resources
        self.ts            = ts
        self.model         = pyo.ConcreteModel()
        self._gross_demand = None   # populated in build()

    @staticmethod
    def _crf(rate, years):
        """Capital Recovery Factor: annualises CAPEX over asset lifetime."""
        r, n = rate, years
        return (r * (1 + r) ** n) / ((1 + r) ** n - 1)

    def build(self):
        r             = self.setup.discount_rate
        techs         = self.setup.selected_gen + self.setup.selected_balancing
        storage_types = self.setup.selected_storage
        T             = range(HOURS_PER_YEAR)
        T_LAST        = HOURS_PER_YEAR - 1

        # ── Gross demand (scaled up by grid loss factor) ───────────────────
        # Ensures installed capacity covers both consumer demand and
        # distribution losses in the island network simultaneously.
        self._gross_demand = self.ts.demand * (1.0 + GRID_LOSS_FACTOR)

        m = self.model
        m.T       = pyo.Set(initialize=T)
        m.Tech    = pyo.Set(initialize=techs)
        m.Storage = pyo.Set(initialize=storage_types)

        # ── Investment variables ───────────────────────────────────────────
        m.Cap              = pyo.Var(m.Tech,    domain=pyo.NonNegativeReals)
        m.StoragePowerCap  = pyo.Var(m.Storage, domain=pyo.NonNegativeReals)
        m.StorageEnergyCap = pyo.Var(m.Storage, domain=pyo.NonNegativeReals)

        # ── Operational variables ──────────────────────────────────────────
        m.RenGen    = pyo.Var(m.Tech,    m.T, domain=pyo.NonNegativeReals)
        m.Curtail   = pyo.Var(m.Tech,    m.T, domain=pyo.NonNegativeReals)
        m.Charge    = pyo.Var(m.Storage, m.T, domain=pyo.NonNegativeReals)
        m.Discharge = pyo.Var(m.Storage, m.T, domain=pyo.NonNegativeReals)
        m.SOC       = pyo.Var(m.Storage, m.T, domain=pyo.NonNegativeReals)
        if "Gas" in techs:
            m.GasGen = pyo.Var(m.T, domain=pyo.NonNegativeReals)

        # ── Capacity limits ────────────────────────────────────────────────
        m.CapLimit = pyo.Constraint(
            m.Tech,
            rule=lambda m, tech: m.Cap[tech] <= self.resources.get(tech)["Max_Capacity_MW"]
        )
        m.StorageCapLimit = pyo.Constraint(
            m.Storage,
            rule=lambda m, s: m.StoragePowerCap[s] <= self.resources.get(s)["Max_Capacity_MW"]
        )

        # ── Most Diversified — minimum installed capacity ─────────────────
        if self.setup.objective == "Most Diversified":
            m.MinCap = pyo.Constraint(
                m.Tech,
                rule=lambda m, tech: m.Cap[tech] >= DIVERSIFIED_MIN_MW
            )
            m.MinStorCap = pyo.Constraint(
                m.Storage,
                rule=lambda m, s: m.StoragePowerCap[s] >= DIVERSIFIED_MIN_MW
            )

        # ── Generation availability ────────────────────────────────────────
        # Profile already represents electrical output per 1 MW installed;
        # efficiency is used only in the economic fuel cost calculation.
        def renewable_balance(m, tech, t):
            if tech == "Gas":
                return pyo.Constraint.Skip
            available = m.Cap[tech] * self.ts.generation[tech][t]
            return m.RenGen[tech, t] + m.Curtail[tech, t] == available
        m.RenewableBalance = pyo.Constraint(m.Tech, m.T, rule=renewable_balance)

        # ── Non-flexible dispatch — curtailment forced to zero ─────────────
        # Biomass, Biogas, Geothermal, and WTE must dispatch at exactly
        # profile(t) × capacity; the zero-curtailment constraint achieves this
        # in combination with the RenewableBalance equality above.
        non_flex_active = [
            tech for tech in techs
            if tech in self.NON_FLEXIBLE and tech in self.ts.generation
        ]
        if non_flex_active:
            m.NonFlexSet     = pyo.Set(initialize=non_flex_active)
            m.NonFlexCurtail = pyo.Constraint(
                m.NonFlexSet, m.T,
                rule=lambda m, tech, t: m.Curtail[tech, t] == 0
            )

        if "Gas" in techs:
            m.GasCapLimit = pyo.Constraint(
                m.T, rule=lambda m, t: m.GasGen[t] <= m.Cap["Gas"]
            )

        # ── Storage power limits ───────────────────────────────────────────
        m.ChargeLimit    = pyo.Constraint(
            m.Storage, m.T,
            rule=lambda m, s, t: m.Charge[s, t]    <= m.StoragePowerCap[s]
        )
        m.DischargeLimit = pyo.Constraint(
            m.Storage, m.T,
            rule=lambda m, s, t: m.Discharge[s, t] <= m.StoragePowerCap[s]
        )

        # ── SoC limits ─────────────────────────────────────────────────────
        m.SOC_Max = pyo.Constraint(
            m.Storage, m.T,
            rule=lambda m, s, t: m.SOC[s, t] <= m.StorageEnergyCap[s]
        )
        m.SOC_Min = pyo.Constraint(
            m.Storage, m.T,
            rule=lambda m, s, t: m.SOC[s, t] >= SOC_MIN_FRACTION * m.StorageEnergyCap[s]
        )

        # ── Storage duration limit ─────────────────────────────────────────
        def duration_limit(m, s):
            max_h = self.setup.max_storage_hours.get(s)
            if max_h is None:
                return pyo.Constraint.Skip
            return m.StorageEnergyCap[s] <= m.StoragePowerCap[s] * max_h
        m.StorageDurationLimit = pyo.Constraint(m.Storage, rule=duration_limit)

        # ── SoC dynamics (cyclic boundary) ─────────────────────────────────
        def soc_rule(m, s, t):
            eff = self.resources.get(s)["Efficiency"]
            if t == 0:
                return m.SOC[s, 0] == SOC_INIT_FRACTION * m.StorageEnergyCap[s]
            return m.SOC[s, t] == m.SOC[s, t-1] + m.Charge[s, t] * eff - m.Discharge[s, t] / eff
        m.SOC_dynamics = pyo.Constraint(m.Storage, m.T, rule=soc_rule)
        m.SOC_cyclic   = pyo.Constraint(
            m.Storage,
            rule=lambda m, s: m.SOC[s, T_LAST] == SOC_INIT_FRACTION * m.StorageEnergyCap[s]
        )

        # ── Storage charges from renewables only ───────────────────────────
        def renewable_charge_limit(m, s, t):
            ren_gen = sum(m.RenGen[tech, t] for tech in m.Tech if tech != "Gas")
            return m.Charge[s, t] <= ren_gen
        m.RenewableChargeLimit = pyo.Constraint(
            m.Storage, m.T, rule=renewable_charge_limit
        )

        # ── Power balance (uses gross demand including grid losses) ────────
        gross_demand = self._gross_demand
        def balance(m, t):
            ren   = sum(m.RenGen[tech, t] for tech in m.Tech if tech != "Gas")
            out   = sum(m.Discharge[s, t] for s in m.Storage)
            inn   = sum(m.Charge[s, t]    for s in m.Storage)
            gas   = m.GasGen[t] if "Gas" in techs else 0
            return ren + out + gas == gross_demand[t] + inn
        m.Balance = pyo.Constraint(m.T, rule=balance)

        # ── Objective ──────────────────────────────────────────────────────
        total_cost = 0

        for tech in techs:
            p   = self.resources.get(tech)
            crf = self._crf(r, p["Lifetime"])

            # ── Fixed annualised cost (CAPEX × CRF + O&M) ─────────────────
            total_cost += m.Cap[tech] * (p["Investment_per_MW"] * crf + p["O&M_per_MW_yr"])

            # ── Variable marginal cost ─────────────────────────────────────
            # Fuel cost converted to €/MWh_electric via thermal efficiency.
            # Carbon shadow price added when Lowest CO2 objective is selected.
            real_mc = p["Fuel_Cost"] / p["Efficiency"] if p["Efficiency"] > 0 else 0.0
            mc = real_mc
            if self.setup.objective == "Lowest CO2":
                mc += p["CO2_per_MWh"] * CARBON_SHADOW_PRICE

            if tech == "Gas":
                total_cost += sum(m.GasGen[t] * mc for t in m.T)
            else:
                if mc > 0:
                    total_cost += sum(m.RenGen[tech, t] * mc for t in m.T)

        for s in storage_types:
            p   = self.resources.get(s)
            crf = self._crf(r, p["Lifetime"])
            total_cost += m.StoragePowerCap[s]  * (p["Investment_per_MW"] * crf + p["O&M_per_MW_yr"])
            total_cost += m.StorageEnergyCap[s] *  p["Storage_MWh"] * crf

        # Curtailment penalty on all non-Gas technologies
        total_cost += sum(
            m.Curtail[tech, t] * CURTAILMENT_PENALTY
            for tech in m.Tech if tech != "Gas" for t in m.T
        )
        # Storage cycling proxy cost
        total_cost += sum(
            m.Charge[s, t] * STORAGE_CHARGE_COST
            for s in m.Storage for t in m.T
        )

        m.Obj = pyo.Objective(expr=total_cost, sense=pyo.minimize)

    def solve(self, tee=True):
        solver  = pyo.SolverFactory("glpk")
        results = solver.solve(self.model, tee=tee)
        status  = results.solver.termination_condition

        if status != pyo.TerminationCondition.optimal:
            raise RuntimeError(
                f"Solver did not find an optimal solution. Status: {status}\n"
                "Check that demand is feasible for the selected technologies."
            )

        obj_label = {
            "Lowest LCOE":      "Lowest LCOE",
            "Lowest CO2":       "Lowest CO\u2082 Emissions",
            "Most Diversified": f"Most Diversified (min {DIVERSIFIED_MIN_MW} MW per technology)",
        }.get(self.setup.objective, self.setup.objective)
        print(f"\nOptimization complete  [{obj_label}].")

        print("\n  Generation capacities:")
        for tech in self.model.Tech:
            print(f"    {tech:<14}: {pyo.value(self.model.Cap[tech]):>8.2f}  MW")
        if list(self.model.Storage):
            print("\n  Storage power capacities:")
            for s in self.model.Storage:
                pwr  = pyo.value(self.model.StoragePowerCap[s])
                ene  = pyo.value(self.model.StorageEnergyCap[s])
                print(f"    {s:<14}: {pwr:>8.2f}  MW  |  {ene:>8.2f}  MWh")

    def export_results(self, filepath="results/optimization_results.xlsx"):
        """
        Export hourly dispatch results to Excel.
        Usage: model.export_results('results/scenario_A.xlsx')
        """
        import os
        m  = self.model
        df = pd.DataFrame(index=range(HOURS_PER_YEAR))

        # ── Generation — actual optimised dispatch (not profile × cap × eff) ─
        for tech in m.Tech:
            if tech == "Gas":
                continue
            df[f"Prod_{tech}_MW"] = [pyo.value(m.RenGen[tech, t]) for t in m.T]
        if "Gas" in m.Tech:
            df["Prod_Gas_MW"] = [pyo.value(m.GasGen[t]) for t in m.T]

        for s in m.Storage:
            df[f"Discharge_{s}_MW"] = [pyo.value(m.Discharge[s, t]) for t in m.T]
            df[f"Charge_{s}_MW"]    = [pyo.value(m.Charge[s, t])    for t in m.T]
            df[f"SOC_{s}_MWh"]      = [pyo.value(m.SOC[s, t])       for t in m.T]

        df["GrossDemand_MW"] = self._gross_demand
        df["NetDemand_MW"]   = self.ts.demand

        os.makedirs(os.path.dirname(filepath) if os.path.dirname(filepath) else ".", exist_ok=True)
        df.to_excel(filepath, index_label="Hour")
        print(f"Results exported to: {filepath}")

    def export_dashboard_json(self, filepath="results/dashboard_results.json", geo=None):
        """
        Export a structured JSON file ready for the HTML dashboard.

        Mirrors the schema produced by the PyPSA-HiGHS notebook so that
        both models can feed the same dashboard without conversion.
        """
        import os, json as _json
        m   = self.model
        r   = self.setup.discount_rate
        cur = self.setup.currency

        gross_demand_mwh = float(sum(self._gross_demand))
        net_demand_mwh   = float(sum(self.ts.demand))

        # ── Meta ──────────────────────────────────────────────────────────
        meta = {
            "objective":        self.setup.objective,
            "discount_rate":    r,
            "currency":         cur,
            "grid_loss_factor": GRID_LOSS_FACTOR,
            "technologies": {
                "generation": self.setup.selected_gen,
                "balancing":  self.setup.selected_balancing,
                "storage":    self.setup.selected_storage,
            },
        }

        # ── Capacities ────────────────────────────────────────────────────
        capacities = {}
        for tech in m.Tech:
            p_res = self.resources.get(tech)
            capacities[tech] = {
                "type":         "generator",
                "capacity_mw":  round(float(pyo.value(m.Cap[tech])), 3),
                "potential_mw": round(float(p_res["Max_Capacity_MW"]), 3),
            }
        for s in m.Storage:
            p_nom = float(pyo.value(m.StoragePowerCap[s]))
            ene   = float(pyo.value(m.StorageEnergyCap[s]))
            mhrs  = self.setup.max_storage_hours.get(s, 4)
            p_res = self.resources.get(s)
            capacities[s] = {
                "type":         "storage",
                "power_mw":     round(p_nom, 3),
                "energy_mwh":   round(ene, 3),
                "max_hours":    mhrs,
                "potential_mw": round(float(p_res["Max_Capacity_MW"]), 3),
            }

        # ── Energy mix + cost component tracking ──────────────────────────
        energy_mix     = {}
        total_ann_cost = 0.0
        total_co2      = 0.0
        _cc            = {}   # cost components per tech

        for tech in m.Tech:
            cap = float(pyo.value(m.Cap[tech]))
            if tech == "Gas":
                gen_mwh = float(sum(pyo.value(m.GasGen[t]) for t in m.T))
            else:
                gen_mwh = float(sum(pyo.value(m.RenGen[tech, t]) for t in m.T))
            gen_gwh = gen_mwh / 1e3
            p       = self.resources.get(tech)
            crf     = self._crf(r, p["Lifetime"])
            real_mc = p["Fuel_Cost"] / p["Efficiency"] if p["Efficiency"] > 0 else 0.0

            ann_capex_t   = cap * p["Investment_per_MW"] * crf
            ann_opex_t    = cap * p["O&M_per_MW_yr"]
            ann_fuel_t    = gen_mwh * real_mc
            total_capex_t = cap * p["Investment_per_MW"]
            ann_c         = ann_capex_t + ann_opex_t + ann_fuel_t

            total_ann_cost += ann_c
            cf    = gen_mwh / (cap * HOURS_PER_YEAR) if cap > 0 else 0.0
            lcoe  = ann_c / gen_mwh if gen_mwh > 0 else None
            co2_t = gen_mwh * float(p.get("CO2_per_MWh", 0.0))
            total_co2 += co2_t

            if tech in self.ts.generation:
                avail_mwh = float(cap * sum(self.ts.generation[tech]))
                curt_gwh  = round(max(0.0, avail_mwh - gen_mwh) / 1e3, 3)
            else:
                curt_gwh  = 0.0

            energy_mix[tech] = {
                "annual_gwh":      round(gen_gwh, 3),
                "share_pct":       round(100 * gen_mwh / gross_demand_mwh, 2) if gross_demand_mwh else 0,
                "capacity_factor": round(cf, 4),
                "lcoe_per_mwh":    round(lcoe, 2) if lcoe is not None else None,
                "annualised_cost": round(ann_c, 0),
                "co2_tco2":        round(co2_t, 1),
                "curtailment_gwh": curt_gwh,
            }
            _cc[tech] = {
                "annualised_capex": round(ann_capex_t, 0),
                "annual_opex":      round(ann_opex_t, 0),
                "annual_fuel_cost": round(ann_fuel_t, 0),
                "total_capex":      round(total_capex_t, 0),
            }

        for s in m.Storage:
            p_nom   = float(pyo.value(m.StoragePowerCap[s]))
            ene     = float(pyo.value(m.StorageEnergyCap[s]))
            dis     = float(sum(pyo.value(m.Discharge[s, t]) for t in m.T))
            chg     = float(sum(pyo.value(m.Charge[s, t])    for t in m.T))
            dis_gwh = dis / 1e3
            chg_gwh = chg / 1e3
            p       = self.resources.get(s)
            crf     = self._crf(r, p["Lifetime"])

            ann_capex_s   = p_nom * p["Investment_per_MW"] * crf + ene * p["Storage_MWh"] * crf
            ann_opex_s    = p_nom * p["O&M_per_MW_yr"]
            total_capex_s = p_nom * p["Investment_per_MW"] + ene * p["Storage_MWh"]
            ann_c         = ann_capex_s + ann_opex_s

            total_ann_cost += ann_c
            lcos = ann_c / dis if dis > 0 else None
            rte  = dis / chg  if chg > 0 else 0.0

            energy_mix[s] = {
                "annual_gwh":      round(dis_gwh, 3),
                "share_pct":       round(100 * dis / gross_demand_mwh, 2) if gross_demand_mwh else 0,
                "discharge_gwh":   round(dis_gwh, 3),
                "charge_gwh":      round(chg_gwh, 3),
                "rte":             round(rte, 4),
                "lcos_per_mwh":    round(lcos, 2) if lcos is not None else None,
                "annualised_cost": round(ann_c, 0),
            }
            _cc[s] = {
                "annualised_capex": round(ann_capex_s, 0),
                "annual_opex":      round(ann_opex_s, 0),
                "annual_fuel_cost": 0,
                "total_capex":      round(total_capex_s, 0),
            }

        # ── LCOE summary (with cost breakdown) ────────────────────────────
        system_lcoe_per_mwh = total_ann_cost / gross_demand_mwh if gross_demand_mwh else 0.0
        lcoe_summary = {}
        for tech in m.Tech:
            cc = _cc.get(tech, {})
            lcoe_summary[tech] = {
                "lcoe_per_mwh":     energy_mix[tech]["lcoe_per_mwh"],
                "annualised_cost":  energy_mix[tech]["annualised_cost"],
                "annualised_capex": cc.get("annualised_capex", 0),
                "annual_opex":      cc.get("annual_opex", 0),
                "annual_fuel_cost": cc.get("annual_fuel_cost", 0),
                "total_capex":      cc.get("total_capex", 0),
            }
        for s in m.Storage:
            cc = _cc.get(s, {})
            lcoe_summary[s] = {
                "lcos_per_mwh":     energy_mix[s]["lcos_per_mwh"],
                "annualised_cost":  energy_mix[s]["annualised_cost"],
                "annualised_capex": cc.get("annualised_capex", 0),
                "annual_opex":      cc.get("annual_opex", 0),
                "annual_fuel_cost": cc.get("annual_fuel_cost", 0),
                "total_capex":      cc.get("total_capex", 0),
            }

        sys_ann_capex   = sum(v["annualised_capex"] for v in _cc.values())
        sys_ann_opex    = sum(v["annual_opex"]      for v in _cc.values())
        sys_ann_fuel    = sum(v["annual_fuel_cost"]  for v in _cc.values())
        sys_total_capex = sum(v["total_capex"]       for v in _cc.values())

        lcoe_summary["_system"] = {
            "system_lcoe_per_mwh":    round(system_lcoe_per_mwh, 2),
            "total_annualised_cost":  round(total_ann_cost, 0),
            "total_demand_mwh":       round(gross_demand_mwh, 1),
            "total_annualised_capex": round(sys_ann_capex, 0),
            "total_annual_opex":      round(sys_ann_opex, 0),
            "total_annual_fuel":      round(sys_ann_fuel, 0),
            "total_capex":            round(sys_total_capex, 0),
        }

        # ── CO₂ summary ───────────────────────────────────────────────────
        co2_summary = {}
        for tech in m.Tech:
            co2_summary[tech] = {"annual_tco2": energy_mix[tech]["co2_tco2"]}
        co2_summary["_system"] = {
            "total_tco2": round(total_co2, 1),
            "emission_intensity_gco2_per_kwh": round(
                total_co2 / net_demand_mwh * 1000 if net_demand_mwh else 0.0, 4
            ),
        }

        # ── Hourly timeseries ─────────────────────────────────────────────
        gross_arr = self._gross_demand
        net_arr   = self.ts.demand

        hourly = []
        for h in range(HOURS_PER_YEAR):
            row = {
                "hour":            h,
                "gross_demand_mw": round(float(gross_arr[h]), 3),
                "net_demand_mw":   round(float(net_arr[h]),   3),
            }
            for tech in m.Tech:
                if tech == "Gas":
                    row["gen_Gas_mw"] = round(float(pyo.value(m.GasGen[h])), 3)
                else:
                    row[f"gen_{tech}_mw"] = round(float(pyo.value(m.RenGen[tech, h])), 3)
            for s in m.Storage:
                row[f"dis_{s}_mw"]  = round(float(pyo.value(m.Discharge[s, h])), 3)
                row[f"chg_{s}_mw"]  = round(float(pyo.value(m.Charge[s, h])),    3)
                row[f"soc_{s}_mwh"] = round(float(pyo.value(m.SOC[s, h])),       3)
            hourly.append(row)

        # ── Geographic (optional) ─────────────────────────────────────────
        geographic = None
        if geo is not None and geo.data is not None:
            row_g = geo.data.iloc[0]
            geographic = {
                "name":         str(row_g.get("Name", "")),
                "latitude":     round(float(row_g["Latitude"]),  6),
                "longitude":    round(float(row_g["Longitude"]), 6),
                "max_height_m": float(row_g["Max_Height_m"]),
            }

        output = {
            "meta":         meta,
            "capacities":   capacities,
            "energy_mix":   energy_mix,
            "lcoe_summary": lcoe_summary,
            "co2_summary":  co2_summary,
            "hourly":       hourly,
        }
        if geographic is not None:
            output["geographic"] = geographic

        os.makedirs(os.path.dirname(filepath) if os.path.dirname(filepath) else ".", exist_ok=True)
        with open(filepath, "w", encoding="utf-8") as fh:
            _json.dump(output, fh, indent=2)
        print(f"\u2705 Dashboard JSON exported \u2192 {filepath}")
        return filepath


model = IslandEnergyMILP(setup, resources, ts)
model.build()
model.solve()


---

## Step 7 — File Export

After solving, the optimised model outputs can be exported to two file
formats for archiving, reporting, and downstream use.

### Excel Export

The `export_results()` method writes a single-sheet Excel workbook with
**8 760 rows** (one per hour) and one column per output timeseries:

| Column pattern | Unit | Description |
|---|---|---|
| `Prod_{tech}_MW` | MW | Hourly generation dispatch per technology |
| `Discharge_{storage}_MW` | MW | Storage discharge (positive flow to grid) |
| `Charge_{storage}_MW` | MW | Storage charge (positive flow from grid) |
| `SOC_{storage}_MWh` | MWh | State of charge at end of each hour |
| `GrossDemand_MW` | MW | Demand including grid losses fed to the model |
| `NetDemand_MW` | MW | Raw consumer demand (before loss factor) |
```python
model.export_results("results/optimization_results.xlsx")
```

### Dashboard JSON Export

The `export_dashboard_json()` method writes a structured JSON file compatible
with the interactive HTML dashboard. Pass `geo=geo` to embed geographic data.

| Key | Contents |
|---|---|
| `meta` | Objective, discount rate, currency, grid loss factor, technology lists |
| `geographic` | Project name, latitude, longitude, max height (only when `geo` is passed) |
| `capacities` | Installed MW (and MWh for storage) + `potential_mw` per technology |
| `energy_mix` | Annual GWh, demand share, capacity factor, LCOE, CO₂, curtailment |
| `lcoe_summary` | Per-technology LCOE/LCOS, annualised cost, CAPEX, OPEX, fuel cost |
| `lcoe_summary._system` | System LCOE, total cost, total demand, system-level cost decomposition |
| `co2_summary` | Per-technology tCO₂/yr, total CO₂, emission intensity (gCO₂/kWh) |
| `hourly` | Full 8 760-row array for all generators, storage, and demand |
```python
model.export_dashboard_json("results/dashboard_results.json", geo=geo)
```

> **Tip:** Edit the filepath arguments to organise results by scenario,
> e.g. `"results/scenario_diversified.xlsx"` and
> `"results/scenario_diversified.json"`.


In [ ]:
model.export_results("results/optimization_results.xlsx")
model.export_dashboard_json("results/dashboard_results.json", geo=geo)


---

## Step 8 — Results Visualization

After solving the optimization problem, results are summarized and visualized
through a comprehensive set of text summaries and charts.

### Text Summaries

| Method | Description |
|--------|-------------|
| `summary()` | Installed capacity of all generation and storage technologies |
| `calculate_lcoe()` | Per-technology LCOE/LCOS, system LCOE, and annual CO₂ summary |

### Charts

| Method | Chart Type | Description |
|--------|-----------|-------------|
| `plot_installed_capacity()` | Bar chart | Installed MW per generation technology |
| `plot_storage_capacity()` | Side-by-side bars | Storage power (MW) and energy (MWh) capacity |
| `plot_energy_mix()` | Donut chart | Annual generation mix as percentage of total demand — callout leader lines |
| `plot_capacity_factors()` | Horizontal bars | Achieved capacity factor per technology |
| `plot_lcoe_breakdown()` | Stacked horizontal bars | Annualised cost decomposed into CAPEX / O&M / fuel per technology |
| `plot_dispatch(hours)` | Stacked area | Hourly dispatch by technology with demand line |
| `plot_load_duration()` | Step chart | Sorted load duration curve with generation breakdown |
| `plot_demand_heatmap()` | Heatmap | Average gross demand by hour-of-day × month (24 × 12 grid) |
| `plot_monthly_cf()` | Heatmap grid | Monthly capacity factor per technology; CF % annotated in each cell |
| `plot_soc(hours)` | Line chart | Storage state of charge with capacity reference bands |
| `plot_residual(hours)` | Area chart | Residual load after renewables (deficit/surplus) |
| `plot_worst_residual_week()` | Combined | Auto-detects and plots the hardest 168-hour window for renewables |
| `plot_energy_sankey()` | Sankey (Plotly) | Interactive annual energy flow: generation → storage → Grid Bus → Load + Losses + Curtailment |

### Dispatch Chart: Stack Order

The `plot_dispatch()` chart stacks technologies in merit order (lowest cost first):

| Layer (bottom → top) | Description |
|---|---|
| WTE, Geothermal, Hydro | Baseload dispatchable renewables |
| Biomass, Biogas | Flexible dispatchable renewables |
| Wind, Solar | Variable renewable generation |
| PHS, BESS, Hydrogen | Storage discharge |
| Gas | Backup balancing generation |

### How to Use

Pass `hours` as a `(start, end)` tuple or a single integer for a
168-hour window starting at that hour:

```python
viz.plot_dispatch((1000, 1168))   # explicit window
viz.plot_dispatch(1000)           # 168-hour window from hour 1000
```


In [ ]:
class ResultsVisualization:
  

    # ── Color palette ──────────────────────────────────────────────────────
    TECH_COLORS = {
        "Wind":        "#4E9AF1",
        "Solar":       "#F4B942",
        "Biomass":     "#2D6A2D",
        "Biogas":      "#6DBF6D",
        "Geothermal":  "#8B4513",
        "Hydro":       "#00AADD",
        "WTE":         "#555555",
        "PHS":         "#00CED1",
        "BESS":        "#3CB371",
        "Gas":         "#8C8C8C",
        "Hydrogen":    "#008B8B",
        "Load":        "#334155",
        "Curtailment": "#E05C5C",
        "Losses":      "#CBD5E1",
    }

    STACK_ORDER = [
        "WTE", "Geothermal", "Hydro", "Biomass", "Biogas",
        "Wind", "Solar", "PHS", "BESS", "Hydrogen", "Gas"
    ]

    def __init__(self, model, setup, resources, ts):
        self.model     = model.model
        self._model    = model         # reference to IslandEnergyMILP for gross_demand
        self.setup     = setup
        self.resources = resources
        self.ts        = ts
        self.cur       = setup.currency   # shorthand used throughout

    # ── Helpers ───────────────────────────────────────────────────────────

    def _val(self, var):
        """Safely extract Pyomo variable value; returns 0 if unavailable."""
        try:
            v = pyo.value(var)
            return v if v is not None else 0.0
        except Exception:
            return 0.0

    def _parse_hours(self, hours):
        """Accept (start, end) tuple or single int (returns 168-hour window)."""
        return hours if isinstance(hours, tuple) else (hours, hours + 168)

    def _color(self, name):
        return self.TECH_COLORS.get(name, "#aaaaaa")

    def _gen_dispatch_all(self, tech):
        """Return full 8760-hour dispatch array for a generation technology."""
        m = self.model
        if tech == "Gas" and hasattr(m, "GasGen"):
            return np.array([self._val(m.GasGen[t]) for t in m.T])
        return np.array([self._val(m.RenGen[tech, t]) for t in m.T])

    def _discharge_all(self, s):
        """Return full 8760-hour discharge array for a storage technology."""
        m = self.model
        return np.array([self._val(m.Discharge[s, t]) for t in m.T])

    @staticmethod
    def _style_ax(ax, title, xlabel="", ylabel="", legend=True):
        """Apply consistent finishing style to an axes object."""
        ax.set_title(title, pad=10)
        if xlabel: ax.set_xlabel(xlabel)
        if ylabel: ax.set_ylabel(ylabel)
        if legend:
            ax.legend(framealpha=0.9, edgecolor="#cccccc",
                      loc="upper right", ncol=2, fontsize=9)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
        )

    # ── Text summaries ────────────────────────────────────────────────────

    def summary(self):
        m   = self.model
        cur = self.cur
        print("\n" + "─"*52)
        print("  INSTALLED GENERATION CAPACITY")
        print("─"*52)
        for tech in m.Tech:
            cap = self._val(m.Cap[tech])
            print(f"  {tech:<14}: {cap:>8.2f}  MW")
        if list(m.Storage):
            print("\n" + "─"*52)
            print("  STORAGE CAPACITY")
            print("─"*52)
            for s in m.Storage:
                pwr = self._val(m.StoragePowerCap[s])
                ene = self._val(m.StorageEnergyCap[s])
                dur = ene / pwr if pwr > 0 else 0
                print(f"  {s}:")
                print(f"    Power capacity  : {pwr:>8.2f}  MW")
                print(f"    Energy capacity : {ene:>8.2f}  MWh")
                print(f"    Duration        : {dur:>8.2f}  h")
        print("─"*52)
        print(f"  Grid loss factor: {GRID_LOSS_FACTOR:.1%}  "
              f"(gross demand = {sum(self.ts.demand)*(1+GRID_LOSS_FACTOR)/1e3:,.1f} GWh/yr)")

    def calculate_lcoe(self):
        m        = self.model
        r        = self.setup.discount_rate
        cur      = self.cur
        total_ac = 0.0

        print("\n" + "─"*62)
        print("  TECHNOLOGY LCOE")
        print("─"*62)
        for tech in m.Tech:
            cap = self._val(m.Cap[tech])
            if cap < 1e-6:
                continue
            p         = self.resources.get(tech)
            crf       = IslandEnergyMILP._crf(r, p["Lifetime"])
            ann_capex = cap * p["Investment_per_MW"] * crf
            ann_om    = cap * p["O&M_per_MW_yr"]
            if tech == "Gas":
                gen_mwh   = sum(self._val(m.GasGen[t]) for t in m.T)
                fuel_cost = sum((self._val(m.GasGen[t]) / p["Efficiency"]) * p["Fuel_Cost"]
                                for t in m.T)
            else:
                gen_mwh   = sum(self._val(m.RenGen[tech, t]) for t in m.T)
                real_mc   = p["Fuel_Cost"] / p["Efficiency"] if p["Efficiency"] > 0 else 0.0
                fuel_cost = real_mc * gen_mwh
            ann_cost  = ann_capex + ann_om + fuel_cost
            total_ac += ann_cost
            if gen_mwh > 0:
                lcoe = ann_cost / gen_mwh
                cf   = gen_mwh / (cap * HOURS_PER_YEAR) if cap > 0 else 0
                print(f"  {tech:<14}: {lcoe:>8.2f}  {cur}/MWh  "
                      f"(gen {gen_mwh/1e3:,.1f} GWh/yr, CF {cf:.1%}, "
                      f"cost {cur}{ann_cost:,.0f}/yr)")

        print("\n" + "─"*62)
        print("  STORAGE LCOS")
        print("─"*62)
        for s in m.Storage:
            p   = self.resources.get(s)
            crf = IslandEnergyMILP._crf(r, p["Lifetime"])
            pwr = self._val(m.StoragePowerCap[s])
            ene = self._val(m.StorageEnergyCap[s])
            ac  = (pwr * p["Investment_per_MW"] * crf
                   + pwr * p["O&M_per_MW_yr"]
                   + ene * p["Storage_MWh"] * crf)
            dis = sum(self._val(m.Discharge[s, t]) for t in m.T)
            chg = sum(self._val(m.Charge[s, t])    for t in m.T)
            rte = dis / chg if chg > 0 else 0
            total_ac += ac
            if dis > 0:
                print(f"  {s:<14}: {ac/dis:>8.2f}  {cur}/MWh  "
                      f"(discharge {dis/1e3:,.1f} GWh/yr, RTE {rte:.1%})")

        total_dem   = sum(self.ts.demand)
        system_lcoe = total_ac / total_dem if total_dem > 0 else 0
        print("\n" + "─"*62)
        print("  SYSTEM LCOE")
        print("─"*62)
        print(f"  Total annualised cost : {cur}{total_ac:>12,.0f}")
        print(f"  Total annual demand   : {total_dem/1e3:>12,.1f}  GWh")
        print(f"  Grid loss factor      : {GRID_LOSS_FACTOR:>11.1%}")
        print(f"  System LCOE           : {system_lcoe:>12.2f}  {cur}/MWh")
        print("─"*62)

        # ── Annual CO2 summary ─────────────────────────────────────────────
        total_co2 = 0.0
        gen_co2   = {}
        for tech in m.Tech:
            p = self.resources.get(tech)
            if tech == "Gas":
                gen_mwh = sum(self._val(m.GasGen[t]) for t in m.T)
            else:
                gen_mwh = sum(self._val(m.RenGen[tech, t]) for t in m.T)
            co2_tech = gen_mwh * p["CO2_per_MWh"]
            if co2_tech > 0:
                gen_co2[tech] = co2_tech
                total_co2    += co2_tech
        if total_co2 > 0:
            print("\n" + "="*60)
            print(" ANNUAL CO2 EMISSIONS")
            print("─"*62)
            for tech, co2 in gen_co2.items():
                print(f"  {tech:<14}: {co2:>10,.1f}  tCO2/yr")
            total_dem = sum(self.ts.demand)
            print(f"  {'TOTAL':<14}: {total_co2:>10,.1f}  tCO2/yr")
            print(f"  Emission intensity: {total_co2/total_dem*1000:>8.1f}  gCO2/kWh")
            print("─"*62)

        return system_lcoe

    # ── Charts ────────────────────────────────────────────────────────────

    def plot_installed_capacity(self):
        """Bar chart: installed generation capacity by technology."""
        m      = self.model
        techs  = list(m.Tech)
        caps   = [self._val(m.Cap[t]) for t in techs]
        colors = [self._color(t) for t in techs]
        fig, ax = plt.subplots(figsize=(9, 5))
        bars = ax.bar(techs, caps, color=colors, edgecolor="white", linewidth=0.8)
        for bar, val in zip(bars, caps):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, val + max(caps) * 0.01,
                        f"{val:.1f}", ha="center", va="bottom", fontsize=9)
        plt.xticks(rotation=30, ha="right")
        ax.set_ylim(0, max(caps) * 1.12)
        self._style_ax(ax, "Installed Generation Capacity", ylabel="MW", legend=False)
        plt.tight_layout()
        plt.show()

    def plot_storage_capacity(self):
        """Side-by-side bars: storage power and energy capacity."""
        m = self.model
        if not list(m.Storage):
            print("No storage technologies selected.")
            return
        storage = list(m.Storage)
        power   = [self._val(m.StoragePowerCap[s])  for s in storage]
        energy  = [self._val(m.StorageEnergyCap[s]) for s in storage]
        colors  = [self._color(s) for s in storage]
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        for ax, vals, unit, title in [
            (axes[0], power,  "MW",  "Storage Power Capacity"),
            (axes[1], energy, "MWh", "Storage Energy Capacity"),
        ]:
            bars = ax.bar(storage, vals, color=colors, edgecolor="white", linewidth=0.8)
            for bar, v in zip(bars, vals):
                if v > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, v + max(vals) * 0.01,
                            f"{v:.1f}", ha="center", va="bottom", fontsize=9)
            self._style_ax(ax, title, ylabel=unit, legend=False)
        plt.tight_layout()
        plt.show()

    def plot_energy_mix(self):
        """
        Donut chart: annual generation mix as share of total demand.
        Arrow tips are anchored to exact wedge midpoints; label positions
        are nudged separately for collision avoidance.
        """
        m            = self.model
        total_demand = sum(self.ts.demand)
        supply       = {}
        for tech in m.Tech:
            if tech == "Gas":
                gas = sum(self._val(m.GasGen[t]) for t in m.T)
                if gas > 1:
                    supply["Gas"] = gas
                continue
            used = sum(self._val(m.RenGen[tech, t]) for t in m.T)
            if used > 1:
                supply[tech] = used
        for s in m.Storage:
            dis = sum(self._val(m.Discharge[s, t]) for t in m.T)
            if dis > 1:
                supply[s] = dis

        labels = list(supply.keys())
        values = [supply[k] / total_demand * 100 for k in labels]
        colors = [self._color(l) for l in labels]

        fig, ax = plt.subplots(figsize=(7, 6.5))
        wedges, _ = ax.pie(
            values,
            labels=None,
            colors=colors,
            startangle=90,
            wedgeprops=dict(width=0.55, edgecolor="white", linewidth=0.8)
        )

        SMALL_SLICE = 5.0
        OUTER_R     = 1.22
        OUTER_R_SM  = 1.48
        TIP_R       = 0.97

        # Read exact wedge midpoint from matplotlib geometry
        label_data = []
        for wedge, lab, val in zip(wedges, labels, values):
            t1, t2 = wedge.theta1, wedge.theta2
            if t2 < t1:
                t2 += 360
            tip_angle = (t1 + t2) / 2
            label_data.append([tip_angle, lab, val, tip_angle])

        label_data.sort(key=lambda x: x[0])
        MIN_SEP = 9.0
        for i in range(1, len(label_data)):
            if label_data[i][0] - label_data[i - 1][0] < MIN_SEP:
                label_data[i][0] = label_data[i - 1][0] + MIN_SEP

        for label_ang, lab, val, tip_ang in label_data:
            tl = np.deg2rad(label_ang)
            cos_l, sin_l = np.cos(tl), np.sin(tl)
            tt = np.deg2rad(tip_ang)
            cos_t, sin_t = np.cos(tt), np.sin(tt)
            r_text = OUTER_R_SM if val < SMALL_SLICE else OUTER_R
            x_text, y_text = r_text * cos_l, r_text * sin_l
            x_tip,  y_tip  = TIP_R  * cos_t, TIP_R  * sin_t
            ha = "left" if cos_l >= 0 else "right"
            ax.annotate(
                f"{lab}\n({val:.1f}%)",
                xy=(x_tip, y_tip),
                xytext=(x_text, y_text),
                ha=ha, va="center", fontsize=9,
                arrowprops=dict(arrowstyle="-", color="#555555", lw=1.0,
                                connectionstyle="arc3,rad=0.0"),
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#e5e7eb",
                          alpha=0.92, lw=0.5)
            )

        ax.set_xlim(-1.9, 1.9)
        ax.set_ylim(-1.9, 1.9)
        ax.set_title("Annual Energy Mix (% of net demand)", pad=16, loc="center")
        plt.tight_layout()
        plt.show()

    def plot_capacity_factors(self):
        """
        Horizontal bar chart of achieved capacity factors by technology.

        Technology names are printed inside each bar; the CF percentage
        is printed to the right. Y-axis tick labels are hidden since
        names appear directly on the bars.
        """
        m      = self.model
        techs  = []; cfs = []; colors = []
        for tech in m.Tech:
            cap = self._val(m.Cap[tech])
            if cap < 1e-3:
                continue
            gen_mwh = sum(self._val(m.GasGen[t]) for t in m.T) if tech == "Gas"                       else sum(self._val(m.RenGen[tech, t]) for t in m.T)
            cf = gen_mwh / (cap * HOURS_PER_YEAR) if cap > 0 else 0.0
            techs.append(tech); cfs.append(cf * 100); colors.append(self._color(tech))

        # Sort by CF descending
        idx    = np.argsort(cfs)[::-1]
        techs  = [techs[i]  for i in idx]
        cfs    = [cfs[i]    for i in idx]
        colors = [colors[i] for i in idx]

        fig, ax = plt.subplots(figsize=(9, max(3.5, len(techs) * 0.72)))
        bars = ax.barh(techs, cfs, color=colors, edgecolor="white", linewidth=0.5)
        for bar, tech, cf in zip(bars, techs, cfs):
            bar_mid = bar.get_y() + bar.get_height() / 2
            bar_w   = bar.get_width()
            ax.text(0.5, bar_mid, tech, ha="left", va="center",
                    fontsize=9, color="white", fontweight="bold")
            ax.text(bar_w + 0.8, bar_mid, f"{cf:.1f}%", va="center",
                    fontsize=9, color="#374151", fontweight=500)
        ax.set_xlim(0, 115)
        ax.axvline(100, color="#d1d5db", linewidth=0.7, linestyle=":", zorder=1)
        ax.set_yticks([])
        self._style_ax(ax, "Achieved Capacity Factor by Technology",
                       xlabel="Capacity Factor (%)", legend=False)
        plt.tight_layout()
        plt.show()

    def plot_lcoe_breakdown(self):
        
        m   = self.model
        r   = self.setup.discount_rate
        cur = self.cur

        techs = [tech for tech in m.Tech if self._val(m.Cap[tech]) > 1e-6]
        capex_vals, om_vals, fuel_vals = [], [], []

        for tech in techs:
            cap = self._val(m.Cap[tech])
            p   = self.resources.get(tech)
            crf = IslandEnergyMILP._crf(r, p["Lifetime"])
            if tech == "Gas":
                gen_mwh = sum(self._val(m.GasGen[t]) for t in m.T)
            else:
                gen_mwh = sum(self._val(m.RenGen[tech, t]) for t in m.T)
            real_mc = p["Fuel_Cost"] / p["Efficiency"] if p["Efficiency"] > 0 else 0.0
            capex_vals.append(cap * p["Investment_per_MW"] * crf / 1e6)
            om_vals.append(   cap * p["O&M_per_MW_yr"] / 1e6)
            fuel_vals.append( real_mc * gen_mwh / 1e6)

        y      = np.arange(len(techs))
        colors = [self._color(t) for t in techs]
        fig, ax = plt.subplots(figsize=(10, max(3.5, len(techs) * 0.68)))
        ax.barh(y, capex_vals, color=colors, edgecolor="white",
                linewidth=0.8, label="CAPEX (annualised)", alpha=0.95)
        ax.barh(y, om_vals, left=capex_vals, color=colors,
                edgecolor="white", linewidth=0.5, label="O&M", alpha=0.60)
        ax.barh(y, fuel_vals,
                left=[c + o for c, o in zip(capex_vals, om_vals)],
                color=colors, edgecolor="white", linewidth=0.5,
                label="Fuel", alpha=0.30)
        for i, tech in enumerate(techs):
            ax.text(0.005, i, tech, ha="left", va="center",
                    fontsize=9, color="white", fontweight="bold")
        ax.set_yticks([])
        self._style_ax(ax, f"Annualised Cost Breakdown by Technology (M{cur}/yr)",
                       xlabel=f"M{cur}/yr", legend=False)
        ax.legend(["CAPEX (annualised)", "O&M", "Fuel"],
                  framealpha=0.9, edgecolor="#cccccc", fontsize=9, loc="lower right")
        plt.tight_layout()
        plt.show()

    def plot_dispatch(self, hours=168):
        """Stacked area dispatch chart in technology merit order."""        
        m          = self.model
        start, end = self._parse_hours(hours)
        T          = range(start, end)
        gen_stack  = []; labels = []; colors = []
        for tech in self.STACK_ORDER:
            if tech in list(m.Tech) and tech != "Gas":
                gen_stack.append(np.array([self._val(m.RenGen[tech, t]) for t in T]))
                labels.append(tech); colors.append(self._color(tech))
            if tech in list(m.Storage):
                gen_stack.append(np.array([self._val(m.Discharge[tech, t]) for t in T]))
                labels.append(f"{tech} (discharge)"); colors.append(self._color(tech))
            if tech == "Gas" and "Gas" in list(m.Tech):
                gen_stack.append(np.array([self._val(m.GasGen[t]) for t in T]))
                labels.append("Gas"); colors.append(self._color("Gas"))
        if not gen_stack:
            print("No generation data to plot.")
            return
        net_demand   = self.ts.demand[start:end]
        gross_demand = net_demand * (1 + GRID_LOSS_FACTOR)
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.stackplot(T, gen_stack, labels=labels, colors=colors, alpha=0.88)
        ax.plot(T, net_demand,   color="#1A1A2E", linewidth=2,   linestyle="--",
                label="Net Demand", zorder=5)
        ax.plot(T, gross_demand, color="#555555", linewidth=1,   linestyle=":",
                alpha=0.6, label="Gross Demand (incl. losses)", zorder=4)
        self._style_ax(ax, f"System Dispatch  (hours {start}–{end})",
                       xlabel="Hour of year", ylabel="MW")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x)}"))
        plt.tight_layout()
        plt.show()

    def plot_load_duration(self):
        """
        Load duration curve (LDC): sorted hourly gross demand with generation
        coverage breakdown.
        """
        m   = self.model
        idx = np.argsort(self.ts.demand)[::-1]
        x   = np.arange(1, HOURS_PER_YEAR + 1)

        stack, labels, colors = [], [], []
        for tech in self.STACK_ORDER:
            if tech in list(m.Tech) and tech != "Gas":
                arr = self._gen_dispatch_all(tech)[idx]
                stack.append(arr); labels.append(tech); colors.append(self._color(tech))
            if tech in list(m.Storage):
                arr = self._discharge_all(tech)[idx]
                stack.append(arr); labels.append(f"{tech} (discharge)"); colors.append(self._color(tech))
            if tech == "Gas" and "Gas" in list(m.Tech):
                arr = self._gen_dispatch_all("Gas")[idx]
                stack.append(arr); labels.append("Gas"); colors.append(self._color("Gas"))

        fig, ax = plt.subplots(figsize=(14, 5))
        if stack:
            ax.stackplot(x, stack, labels=labels, colors=colors, alpha=0.82)
        demand_sorted = self.ts.demand[idx]
        ax.plot(x, demand_sorted, color="#1E293B", linewidth=2.2, linestyle="--",
                label="Net Demand", zorder=5)
        ax.plot(x, demand_sorted * (1 + GRID_LOSS_FACTOR), color="#555555",
                linewidth=1, linestyle=":", alpha=0.6, label="Gross Demand", zorder=4)
        self._style_ax(ax, "Load Duration Curve — Annual Dispatch (sorted by demand)",
                       xlabel="Hours (ranked, peak to valley)", ylabel="MW")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
        plt.tight_layout()
        plt.show()

    def plot_demand_heatmap(self):
        """
        24h × 12mo heatmap of average hourly gross demand.

        Rows are hour-of-day (0–23), columns are months (Jan–Dec).
        Colour intensity maps demand magnitude — reveals diurnal and
        seasonal load structure at a glance.
        """
        demand = self.ts.demand * (1 + GRID_LOSS_FACTOR)
        hm  = np.zeros((24, 12))
        cnt = np.zeros((24, 12))
        months_idx = np.searchsorted(
            np.cumsum([744, 672, 744, 720, 744, 720, 744, 744, 720, 744, 720, 744]),
            np.arange(HOURS_PER_YEAR), side="right"
        )
        for h in range(HOURS_PER_YEAR):
            hod, mo = h % 24, min(months_idx[h], 11)
            hm[hod, mo] += demand[h]
            cnt[hod, mo] += 1
        cnt[cnt == 0] = 1
        hm /= cnt

        fig, ax = plt.subplots(figsize=(9, 5.5))
        im = ax.imshow(hm, aspect="auto", cmap="YlOrRd", interpolation="nearest")
        ax.set_xticks(range(12))
        ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                            "Jul","Aug","Sep","Oct","Nov","Dec"])
        ax.set_yticks(range(0, 24, 3))
        ax.set_yticklabels([f"{h:02d}:00" for h in range(0, 24, 3)])
        ax.set_xlabel("Month")
        ax.set_ylabel("Hour of day")
        cbar = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
        cbar.set_label("MW", fontsize=9)
        cbar.ax.tick_params(labelsize=8)
        self._style_ax(ax, "Demand Heatmap — Hour of Day × Month", legend=False)
        ax.grid(False)
        plt.tight_layout()
        plt.show()

    def plot_monthly_cf(self):
        """
        Monthly capacity factor grid: technology × month.

        Each cell shows the achieved CF for that technology in that month,
        with colour intensity mapped to utilisation level. Gives a compact
        view of seasonal performance across the entire technology portfolio.
        """
        m = self.model
        months_idx = np.searchsorted(
            np.cumsum([744, 672, 744, 720, 744, 720, 744, 744, 720, 744, 720, 744]),
            np.arange(HOURS_PER_YEAR), side="right"
        )
        hrs_per_month = np.array([744, 672, 744, 720, 744, 720, 744, 744, 720, 744, 720, 744])

        techs = [tech for tech in m.Tech if self._val(m.Cap[tech]) > 1e-3]
        if not techs:
            print("No generation technologies to plot.")
            return

        cf_matrix = np.zeros((len(techs), 12))
        for i, tech in enumerate(techs):
            cap      = self._val(m.Cap[tech])
            dispatch = self._gen_dispatch_all(tech)
            for h in range(HOURS_PER_YEAR):
                mo = min(months_idx[h], 11)
                cf_matrix[i, mo] += dispatch[h]
            for mo in range(12):
                cf_matrix[i, mo] = cf_matrix[i, mo] / (cap * hrs_per_month[mo]) * 100

        fig, ax = plt.subplots(figsize=(9, max(2.5, len(techs) * 0.55 + 1)))
        im = ax.imshow(cf_matrix, aspect="auto", cmap="GnBu",
                       interpolation="nearest", vmin=0, vmax=100)
        ax.set_xticks(range(12))
        ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                            "Jul","Aug","Sep","Oct","Nov","Dec"])
        ax.set_yticks(range(len(techs)))
        ax.set_yticklabels(techs)
        for i in range(len(techs)):
            for j in range(12):
                val   = cf_matrix[i, j]
                color = "white" if val > 50 else "#374151"
                ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                        fontsize=8, color=color, fontweight=500)
        cbar = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
        cbar.set_label("CF %", fontsize=9)
        cbar.ax.tick_params(labelsize=8)
        self._style_ax(ax, "Monthly Capacity Factor by Technology", legend=False)
        ax.grid(False)
        plt.tight_layout()
        plt.show()

    def plot_soc(self, hours=168):
        """Battery state of charge with capacity reference bands."""
        m = self.model
        if not list(m.Storage):
            print("No storage technologies selected.")
            return
        start, end = self._parse_hours(hours)
        T          = range(start, end)
        fig, ax    = plt.subplots(figsize=(14, 4.5))
        for s in m.Storage:
            soc = np.array([self._val(m.SOC[s, t]) for t in T])
            cap = self._val(m.StorageEnergyCap[s])
            ax.fill_between(T, soc, alpha=0.25, color=self._color(s))
            ax.plot(T, soc, color=self._color(s), linewidth=2, label=s)
            ax.axhline(cap,                    color=self._color(s), linestyle=":",  linewidth=1, alpha=0.6)
            ax.axhline(cap * SOC_MIN_FRACTION, color="#E05C5C",      linestyle="--", linewidth=1, alpha=0.7)
        self._style_ax(ax, f"Storage State of Charge  (hours {start}–{end})",
                       xlabel="Hour of year", ylabel="MWh")
        plt.tight_layout()
        plt.show()

    def plot_residual(self, hours=168):
        """Residual load chart: demand minus all non-storage generation."""
        m          = self.model
        start, end = self._parse_hours(hours)
        T          = range(start, end)
        ren        = np.zeros(end - start)
        for tech in m.Tech:
            if tech == "Gas":
                continue
            ren += np.array([self._val(m.RenGen[tech, t]) for t in T])
        residual = self.ts.demand[start:end] - ren
        fig, ax  = plt.subplots(figsize=(14, 3.5))
        ax.plot(T, residual, color="#1A1A2E", linewidth=1.2)
        ax.axhline(0, color="#888888", linestyle="--", linewidth=0.8)
        ax.fill_between(T, residual, 0,
                        where=(residual > 0),  color="#E05C5C", alpha=0.4,
                        label="Deficit (storage / gas needed)")
        ax.fill_between(T, residual, 0,
                        where=(residual <= 0), color="#3CB371", alpha=0.3,
                        label="Surplus (curtailment / storage charge)")
        self._style_ax(ax, f"Residual Load  (hours {start}–{end})",
                       xlabel="Hour of year", ylabel="MW", legend=True)
        ax.legend(loc="upper right", ncol=1, fontsize=9)
        plt.tight_layout()
        plt.show()

    def plot_worst_residual_week(self):
        """
        Identify and plot the week with the highest cumulative residual load.
        Uses fossil/backup generation as the measure when available; otherwise
        falls back to residual load against renewables.
        This is the most critical week for storage and gas backup adequacy.
        """
        m      = self.model
        WEEK   = 168
        FOSSIL = {"Gas"}

        fossils = [tech for tech in m.Tech if tech in FOSSIL]
        if fossils:
            # Sliding window over backup generation hours
            measure = np.array([
                sum(self._val(m.GasGen[t]) for tech in fossils)
                for t in range(HOURS_PER_YEAR)
            ])
        else:
            # Fall back: residual load (demand minus all generation)
            ren = np.zeros(HOURS_PER_YEAR)
            for tech in m.Tech:
                if tech != "Gas":
                    ren += self._gen_dispatch_all(tech)
            measure = np.array(self.ts.demand) - ren

        win = sum(measure[:WEEK])
        best = win; worst_start = 0
        for i in range(1, HOURS_PER_YEAR - WEEK):
            win -= measure[i - 1]; win += measure[i + WEEK - 1]
            if win > best:
                best = win; worst_start = i

        print(f"\nWorst renewable week: hours {worst_start}–{worst_start + WEEK}")
        print(f"  Calendar week ≈ {worst_start // 168 + 1}")
        self.plot_dispatch((worst_start, worst_start + WEEK))
        self.plot_soc((worst_start, worst_start + WEEK))
        self.plot_residual((worst_start, worst_start + WEEK))

    def plot_energy_sankey(self, title="Annual Energy Flow (GWh/yr)"):
        """
        Interactive Sankey diagram of annual energy flows.

        Column layout (left → right)
        ─────────────────────────────
        Col 1  Generation nodes  (one per active generator)
        Col 2  Storage nodes     (one per active storage unit, if any)
        Col 3  Grid Bus          (single balancing node)
        Col 4  Sinks             Load · Losses · Curtailment

        Flow architecture
        ─────────────────
        1. Generator  →  Storage       pro-rata share of total charging
        2. Generator  →  Grid Bus      remainder after storage allocation
        3. Generator  →  Curtailment   flexible / VRE techs only
        4. Storage    →  Grid Bus      discharge
        5. Storage    →  Losses        round-trip loss (charge − discharge)
        6. Grid Bus   →  Load          net consumer demand
        7. Grid Bus   →  Losses        distribution loss (GLF × gross demand)

        Storage and grid losses are merged into a single Losses sink node.
        Chart height scales automatically with number of generator technologies.
        """
        m             = self.model
        gen_techs     = list(m.Tech)
        storage_types = list(m.Storage)
        has_storage   = len(storage_types) > 0

        # ── Annual totals ─────────────────────────────────────────────────
        gen_dispatched = {}
        gen_curtailed  = {}
        for tech in gen_techs:
            if tech == "Gas":
                gen_dispatched[tech] = float(sum(self._val(m.GasGen[t]) for t in m.T))
                gen_curtailed[tech]  = 0.0
            else:
                gen_dispatched[tech] = float(sum(self._val(m.RenGen[tech, t]) for t in m.T))
                gen_curtailed[tech]  = float(sum(self._val(m.Curtail[tech, t]) for t in m.T))

        stor_charge    = {s: float(sum(self._val(m.Charge[s, t])    for t in m.T)) for s in storage_types}
        stor_discharge = {s: float(sum(self._val(m.Discharge[s, t]) for t in m.T)) for s in storage_types}
        stor_loss      = {s: max(stor_charge[s] - stor_discharge[s], 0.) for s in storage_types}

        total_dispatched = sum(gen_dispatched.values())
        total_charge     = sum(stor_charge.values())
        net_demand_mwh   = float(sum(self.ts.demand))
        gross_demand_mwh = net_demand_mwh * (1 + GRID_LOSS_FACTOR)
        grid_loss_mwh    = gross_demand_mwh - net_demand_mwh

        # ── Node registry ─────────────────────────────────────────────────
        node_labels, node_colors, node_x, node_y = [], [], [], []

        def add_node(label, color, x, y=0.5):
            node_labels.append(label); node_colors.append(color)
            node_x.append(x);         node_y.append(y)
            return len(node_labels) - 1

        X_GEN  = 0.02
        X_STOR = 0.36
        X_GRID = 0.62 if has_storage else 0.48
        X_SINK = 0.98

        n_gen  = len(gen_techs)
        gen_idx = {}
        for i, g in enumerate(gen_techs):
            y_pos = (i + 0.5) / n_gen if n_gen > 1 else 0.5
            gen_idx[g] = add_node(g, self._color(g), X_GEN, y_pos)

        stor_idx = {}
        if has_storage:
            n_stor = len(storage_types)
            for i, s in enumerate(storage_types):
                y_pos = (i + 0.5) / n_stor if n_stor > 1 else 0.5
                stor_idx[s] = add_node(s, self._color(s), X_STOR, y_pos)

        grid_idx = add_node("Grid Bus",    "#94A3B8", X_GRID, 0.45)
        load_idx = add_node("Load",        "#334155", X_SINK, 0.28)
        loss_idx = add_node("Losses",      "#CBD5E1", X_SINK, 0.72)
        curt_idx = add_node("Curtailment", "#E05C5C", X_SINK, 0.92)

        # ── Link registry ─────────────────────────────────────────────────
        link_src, link_tgt, link_val, link_col, link_lbl = [], [], [], [], []

        def add_link(src, tgt, mwh, color_name, label):
            if mwh < 1.0:
                return
            link_src.append(src); link_tgt.append(tgt)
            link_val.append(round(mwh / 1000, 2))
            link_lbl.append(label)
            base   = self._color(color_name).lstrip("#")
            r, g, b = int(base[0:2], 16), int(base[2:4], 16), int(base[4:6], 16)
            link_col.append(f"rgba({r},{g},{b},0.35)")

        # ── Generator flows ───────────────────────────────────────────────
        for tech in gen_techs:
            disp  = gen_dispatched[tech]
            curt  = gen_curtailed[tech]
            share = disp / total_dispatched if total_dispatched > 0 else 0.0

            if has_storage:
                for s in storage_types:
                    alloc = share * stor_charge[s]
                    add_link(gen_idx[tech], stor_idx[s], alloc, tech,
                             f"{tech} \u2192 {s} charging  {alloc/1e3:.2f} GWh")

            stor_alloc_total = share * total_charge
            to_grid          = max(disp - stor_alloc_total, 0.0)
            add_link(gen_idx[tech], grid_idx, to_grid, tech,
                     f"{tech} \u2192 Grid Bus  {to_grid/1e3:.2f} GWh")
            add_link(gen_idx[tech], curt_idx, curt, tech,
                     f"{tech} \u2192 Curtailment  {curt/1e3:.2f} GWh")

        # ── Storage flows ─────────────────────────────────────────────────
        for s in storage_types:
            dis  = stor_discharge[s]
            loss = stor_loss[s]
            rte  = dis / stor_charge[s] * 100 if stor_charge[s] > 0 else 0.0
            add_link(stor_idx[s], grid_idx, dis, s,
                     f"{s} discharge \u2192 Grid Bus  {dis/1e3:.2f} GWh  (\u03b7 {rte:.0f}%)")
            add_link(stor_idx[s], loss_idx, loss, s,
                     f"{s} round-trip loss  {loss/1e3:.2f} GWh")

        # ── Grid Bus → sinks ──────────────────────────────────────────────
        add_link(grid_idx, load_idx, net_demand_mwh, "Load",
                 f"Grid Bus \u2192 Load  {net_demand_mwh/1e3:.2f} GWh")
        add_link(grid_idx, loss_idx, grid_loss_mwh, "Losses",
                 f"Grid distribution loss  {grid_loss_mwh/1e3:.2f} GWh  ({GRID_LOSS_FACTOR*100:.0f}%)")

        # ── Summary subtitle ──────────────────────────────────────────────
        total_curt_gwh = sum(gen_curtailed.values()) / 1e3
        total_dis_gwh  = sum(stor_discharge.values()) / 1e3
        total_sl_gwh   = sum(stor_loss.values()) / 1e3
        total_loss_gwh = grid_loss_mwh / 1e3 + total_sl_gwh

        subtitle = (
            f"Dispatched: {total_dispatched/1e3:.1f} GWh  \u2502  "
            f"Net demand: {net_demand_mwh/1e3:.1f} GWh  \u2502  "
            f"Total losses: {total_loss_gwh:.1f} GWh  \u2502  "
            f"Storage discharge: {total_dis_gwh:.1f} GWh  \u2502  "
            f"Curtailment: {total_curt_gwh:.1f} GWh"
        )

        dynamic_height = max(520, 90 * max(n_gen, len(storage_types) if has_storage else 1) + 140)

        # ── Build figure ──────────────────────────────────────────────────
        fig = go.Figure(go.Sankey(
            arrangement="snap",
            node=dict(
                label=node_labels, color=node_colors,
                x=node_x, y=node_y,
                pad=18, thickness=20,
                line=dict(color="rgba(255,255,255,0.6)", width=0.5),
                hovertemplate=(
                    "<b>%{label}</b><br>"
                    "Total flow: %{value:,.2f} GWh"
                    "<extra></extra>"
                ),
            ),
            link=dict(
                source=link_src, target=link_tgt, value=link_val,
                label=link_lbl,  color=link_col,
                hovertemplate="%{label}<extra></extra>",
            ),
        ))
        fig.update_layout(
            title=dict(
                text=(
                    f"<b>{title}</b><br>"
                    f"<span style='font-size:12px;color:#888'>{subtitle}</span>"
                ),
                font=dict(size=16),
                x=0.01, xanchor="left",
            ),
            font=dict(family="Arial, sans-serif", size=11.5, color="#374151"),
            paper_bgcolor="white",
            plot_bgcolor="white",
            height=dynamic_height,
            margin=dict(l=20, r=20, t=110, b=30),
        )
        fig.show()


In [ ]:
# ── Run all results ────────────────────────────────────────────────────────────
viz = ResultsVisualization(model, setup, resources, ts)

viz.summary()
viz.calculate_lcoe()

viz.plot_energy_mix()
viz.plot_installed_capacity()
viz.plot_storage_capacity()
viz.plot_capacity_factors()
viz.plot_lcoe_breakdown()
viz.plot_dispatch((4000, 4300))    # adjust window as needed
viz.plot_load_duration()
viz.plot_demand_heatmap()
viz.plot_monthly_cf()
viz.plot_energy_sankey()
viz.plot_soc(1000)
viz.plot_residual(1000)

viz.plot_worst_residual_week()


---

## Model Summary

This notebook demonstrates a **techno-economic optimization** of a renewable-based
island energy system using Mixed Integer Linear Programming (MILP), implemented
in the Pyomo optimization modelling framework and solved with the GLPK solver.

The model determines the **least-cost system configuration** of:

- **Generation capacity** — optimal installed MW for each technology
- **Storage capacity** — optimal power (MW) and energy (MWh) sizing
- **Operational dispatch** — hourly scheduling of generation and storage
- **Grid losses** — distribution loss factor applied to gross demand

while satisfying hourly electricity demand and all technical constraints,
and minimising total annualised system cost or CO₂ emissions.

---

### Potential Applications

| Application | Description |
|---|---|
| Renewable energy planning | Optimal technology mix for island or remote systems |
| Microgrid design | Off-grid or grid-connected community energy systems |
| Island power systems | Replacement of diesel with renewables |
| Storage evaluation | Role and sizing of BESS, PHS, and hydrogen |
| Renewable integration | Analysis of high-VRE systems with storage |
| Academic research | Teaching and research in energy systems modelling |

---

### Potential Extensions

The model is designed to be **modular and extensible**. Future versions could include:

| Extension | Description |
|---|---|
| Stochastic optimization | Uncertainty in demand, wind, and solar resources |
| Multi-year investment | Phased capacity expansion over a planning horizon |
| Transmission constraints | Network-constrained dispatch for multi-node systems |
| CO₂ budget constraint | Hard emission cap alongside cost minimization |
| Demand response | Flexible loads as a system balancing resource |
| EV fleet integration | Electric vehicles as distributed storage |

---

### References

- Hart, W.E. et al. (2017). *Pyomo – Optimization Modeling in Python*. Springer.
- IRENA (2023). *Renewable Power Generation Costs*. International Renewable Energy Agency.
- IEA (2023). *World Energy Outlook*. International Energy Agency.
- NREL (2023). *Annual Technology Baseline*. National Renewable Energy Laboratory.
- Pfenninger, S. & Staffell, I. (2016). Long-term patterns of European PV output.
  *Energy*, 114, 1251–1265. [Renewables.ninja]
- Anthropic (2025). *Claude AI Assistant* (claude.ai). Used to support model development,
  code review, and documentation.

---

*Developed by Agus Samsudin — Energy Systems Modelling*
